In [1]:
# Question 1: Install Spark and PySpark

In [39]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql import types

In [3]:
# Create a Local Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/09/24 12:16:57 WARN Utils: Your hostname, GANIU-ODEYINKA resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/24 12:16:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/24 12:16:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Execute spark.version
pyspark.__version__

'3.5.2'

In [5]:
# Question 2: Yellow October 2024
# wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
# Read the October 2024 Yellow into a Spark Dataframe.Repartition the Dataframe to 4 partitions and save it to parquet.answer which most closely matches.

In [6]:
#!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

In [7]:
df = spark.read.parquet('yellow_tripdata_2024-10.parquet')

In [8]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [9]:
df = df.repartition(4)
df.write.parquet('data-output')

In [10]:
!pwd

/home/iamomowale/my_DE_Zoomcamp_2025/05-batch/homework


In [11]:
!ls -lh /home/iamomowale/my_DE_Zoomcamp_2025/05-batch/homework/data-output

total 90M
-rw-r--r-- 1 iamomowale iamomowale   0 Sep 24 12:18 _SUCCESS
-rw-r--r-- 1 iamomowale iamomowale 23M Sep 24 12:18 part-00000-37a8d0cc-3e14-4c9a-a387-47b4ffd2131c-c000.snappy.parquet
-rw-r--r-- 1 iamomowale iamomowale 23M Sep 24 12:18 part-00001-37a8d0cc-3e14-4c9a-a387-47b4ffd2131c-c000.snappy.parquet
-rw-r--r-- 1 iamomowale iamomowale 23M Sep 24 12:18 part-00002-37a8d0cc-3e14-4c9a-a387-47b4ffd2131c-c000.snappy.parquet
-rw-r--r-- 1 iamomowale iamomowale 23M Sep 24 12:18 part-00003-37a8d0cc-3e14-4c9a-a387-47b4ffd2131c-c000.snappy.parquet


In [12]:
df.createOrReplaceTempView('homework')

In [13]:
# Question 3: Count records
# How many taxi trips were there on the 15th of October? Consider only trips that started on the 15th of October.

In [14]:
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2024-10-15'") \
    .count()

128893

In [26]:
spark.sql("""
    SELECT 
        count(*)
    FROM
        homework
    WHERE
        cast(tpep_pickup_datetime as date) = '2024-10-15' and tpep_pickup_datetime is not null;
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [ ]:
#Question 4: Longest trip
#What is the length of the longest trip in the dataset in hours?

In [34]:
df \
    .withColumn("duration", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600) \
    .orderBy('duration', ascending=False) \
    .limit(5) \
    .select('duration') \
    .show()

[Stage 48:>                                                         (0 + 4) / 4]

+------------------+
|          duration|
+------------------+
|162.61777777777777|
|           143.325|
|137.76055555555556|
|114.83472222222223|
| 89.89833333333333|
+------------------+



In [37]:
spark.sql("""
    SELECT 
        (TIMESTAMPDIFF(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600) AS trip_duration_seconds
    FROM 
        homework
    ORDER BY 
        trip_duration_seconds DESC
    LIMIT 5;
""").show()

[Stage 57:>                                                         (0 + 4) / 4]

+---------------------+
|trip_duration_seconds|
+---------------------+
|   162.61777777777777|
|              143.325|
|   137.76055555555556|
|   114.83472222222223|
|    89.89833333333333|
+---------------------+



In [63]:
# Question 5: User Interface
# Spark’s User Interface which shows the application's dashboard runs on which local port?

# Answer: 4040

In [48]:
# Question 6: Least frequent pickup location zone
# Load the zone lookup data into a temp view in Spark: wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
# Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

In [49]:
#!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [40]:
taxi_zone_schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True)
])

In [44]:
df_zones = spark.read \
    .option("header", "true") \
    .schema(taxi_zone_schema) \
    .csv('./taxi_zone_lookup.csv')

In [47]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [52]:
df_zones.createOrReplaceTempView("zones")

In [62]:
spark.sql("""
    SELECT 
        zones.Zone,
        count(1) AS count
    FROM 
        homework
    LEFT JOIN
        zones ON zones.LocationID = homework.PULocationID
    GROUP BY
        1
    ORDER BY 
        2
    LIMIT 5;
""").show()

[Stage 96:=============================>                            (2 + 2) / 4]

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Rikers Island|    2|
|       Arden Heights|    2|
|         Jamaica Bay|    3|
| Green-Wood Cemetery|    3|
+--------------------+-----+



In [64]:
spark.stop()